# Dataset A: llimllib/nba_data, Regular-Season Player Stats (2015-16 to 2025-26)

This notebook builds the **regular-season** player features from `ListOfPotentialFeatures.md`:

| Feature | Column(s) | Level |
|---|---|---|
| Regular-season PPG, RPG, APG, BPG, SPG | `rs_ppg`, `rs_rpg`, `rs_apg`, `rs_bpg`, `rs_spg` | player-season |
| Regular-season true shooting % | `rs_ts_pct` | player-season |
| Regular-season games played, minutes per game | `rs_gp`, `rs_mpg` | player-season |
| Age, years in the NBA | `age`, `years_in_nba` | player-season |
| 3-year trend: PPG, RPG, APG, minutes, games played | `avg_ppg_last_3`, `avg_rpg_last_3`, `avg_apg_last_3`, `avg_mpg_last_3`, `avg_gp_last_3` (+ `seasons_played_last_3`) | player-season |
| (extra) on-court plus/minus per game | `rs_plus_minus` | player-season |

**Regular-season Box +/-** (and its 3-year trend) comes from Basketball-Reference in notebook `03`, because this database has no Box Plus/Minus (concern 1). Playoff stats are in notebook `02`. Notebook `999` combines all three into one table.

The output is **one row per player + season**, joined onto the salary table on `season` + `nba_player_id`. **Why the current season's stats are usable:** we only predict **offseason** trades, so each season's stats are final before the trade window opens.

*Run order: `01` → `02` → `03` → `999`. Notebook 02 reads the raw files this notebook downloads.*

- **Dataset name:** llimllib/nba_data, a public DuckDB/parquet catalog of NBA data rebuilt every 4 hours (the same source as `salary-data/02` and `team-performance/01`)
- **Link:** https://github.com/llimllib/nba_data (data is served from `https://basketball-data.billmill.org/nba/nba.duckdb`)
- **Tables used:**
  - `player_season_stats`: NBA.com season **totals** for each player, one row per player per season per `season_type` (`regular_season` or `playoffs`).
  - `players`: NBA `player_id` → name.
  - `team_seasons`: NBA `team_id` → 3-letter abbreviation, for the player's end-of-season team.
- **Saved as:** `data-files/00-raw/llimllib_player_season_stats_raw.csv`, `llimllib_players_raw.csv`, `llimllib_team_seasons_raw.csv` (downloaded below if missing; the last two are the same files `salary-data/02` downloads)

## Important variables

| Variable | Units | Meaning |
|---|---|---|
| `season` | year | The year the season **ends** (2026 = 2025-26). Same convention as our other notebooks. |
| `season_type` | text | `regular_season` or `playoffs`. This notebook uses `regular_season`; notebook 02 uses `playoffs`. |
| `player_id` | text | NBA.com person id (e.g. `201939` = Stephen Curry). Renamed `nba_player_id`, the join key with the salary table. |
| `gp`, `min` | games, minutes | Games played and **total** minutes in the season. |
| `pts`, `reb`, `ast`, `stl`, `blk` | counts | Season **totals**. Divided by `gp` to get per-game stats. |
| `fga`, `fta` | counts | Field-goal and free-throw attempts, used to check `ts_pct`. |
| `ts_pct` | 0-1 (can exceed 1) | True shooting %: `pts / (2 × (fga + 0.44 × fta))`. |
| `plus_minus` | points | Total on-court plus/minus: the team's point margin while the player was on the floor. |
| `team_count`, `player_last_team_id` | count, id | How many teams the player played for that season, and his last one. |
| `draft_year` | year | Year drafted. Blank for undrafted players. |

## Concerns going in

1. **The database has no Box Plus/Minus.** Its `plus_minus` is the on-court point margin, which mostly measures the player's team and teammates. Box Plus/Minus (BPM) is a box-score estimate of a player's own impact. So BPM comes from Basketball-Reference in notebook 03, and we keep on-court plus/minus only as an extra column.
2. **Are traded players counted once or once per team?** If per team, their per-game stats would need to be combined. We test this in section 3A.
3. **The 3-year trends need earlier seasons.** Each trend is the average over the **three seasons before** this one (the current season is not included, the same as the team 5-year trends). So for 2015-16 we also load **2012-13 through 2014-15**. They are used only for the trends and are not rows in the output.
4. **Small samples.** Players with a handful of games have noisy per-game stats. We **keep every row** rather than dropping low-game players: the rows are joined onto the salary table, and dropping one would also blank that player's age, experience, and games played. `rs_gp` and `rs_mpg` tell the model how much to trust the per-game numbers.
5. **Years in the NBA.** Counting from the draft year overcounts players who were drafted but joined the league later ("draft-and-stash"). We test a count from each player's first season in the stats table instead (section 3E).

## Setup

In [1]:
import sys
from pathlib import Path

import duckdb
import numpy as np
import pandas as pd

# The shared helpers (paths, season range, name keys, prior-season averages) live next to the salary notebooks.
sys.path.insert(0, str(Path('..') / 'salary-data'))
from nba_cleaning_utils import RAW, PROCESSED, FIRST_SEASON, LAST_SEASON, normalize_name, prior_seasons_average

# The 3-year averages for FIRST_SEASON need the three seasons before it.
HISTORY_FIRST = FIRST_SEASON - 3
print('output seasons:', FIRST_SEASON, 'to', LAST_SEASON, '| history loaded from:', HISTORY_FIRST)

output seasons: 2016 to 2026 | history loaded from: 2013


## 1. Download (skipped if the raw files already exist)

DuckDB reads the remote catalog over HTTPS and downloads only the parquet files a query needs. We save the raw tables as CSV so the rest of the notebook runs offline. We download every season and filter in pandas below.

In [2]:
RAW_STATS = RAW / 'llimllib_player_season_stats_raw.csv'
RAW_PLAYERS = RAW / 'llimllib_players_raw.csv'
RAW_TEAMS = RAW / 'llimllib_team_seasons_raw.csv'

queries = {
    RAW_STATS: '''SELECT season, season_type, player_id, player_last_team_id, team_count, age, gp, min,
                         pts, reb, ast, stl, blk, fga, fta, plus_minus, ts_pct, draft_year
                  FROM nba.player_season_stats ORDER BY season, season_type, player_id''',
    RAW_PLAYERS: 'SELECT * FROM nba.players ORDER BY player_id',
    RAW_TEAMS: 'SELECT * FROM nba.team_seasons ORDER BY season, team_id',
}
missing = [p for p in queries if not p.exists()]
if missing:
    con = duckdb.connect()
    con.sql('INSTALL httpfs; LOAD httpfs;')
    con.sql("ATTACH 'https://basketball-data.billmill.org/nba/nba.duckdb' AS nba (READ_ONLY)")
    for path in missing:
        con.sql(queries[path]).df().to_csv(path, index=False)
        print('downloaded', path.name)
else:
    print('raw files already present, skipping download')

raw files already present, skipping download


## 2. Load the data

In [3]:
ids = {'player_id': str, 'player_last_team_id': str, 'team_id': str}
stats_raw = pd.read_csv(RAW_STATS, dtype=ids)
players = pd.read_csv(RAW_PLAYERS, dtype=ids)
team_seasons = pd.read_csv(RAW_TEAMS, dtype=ids)

rs_raw = stats_raw[stats_raw['season_type'] == 'regular_season'].copy()
rs_raw.head()

,season,season_type,player_id,player_last_team_id,team_count,age,gp,min,pts,reb,ast,stl,blk,fga,fta,plus_minus,ts_pct,draft_year
189,1997,regular_season,100,NaN,1,30.0,15,182.410000,44,21,7,3,5,48,7,-10,0.431,NaN
190,1997,regular_season,1000,NaN,1,23.0,65,1066.818333,386,179,49,27,8,318,99,37,0.534,1996.0
191,1997,regular_season,1002,NaN,1,23.0,39,240.495000,57,15,36,13,2,58,11,19,0.454,1996.0
192,1997,regular_season,1005,NaN,1,27.0,73,2649.415000,1199,367,197,97,62,982,243,-53,0.551,1992.0
193,1997,regular_season,1006,NaN,1,39.0,21,183.725000,39,31,5,4,5,46,4,-37,0.408,1981.0


In [4]:
players.head(3)

,player_id,name,first_season,last_season
0,101106,Andrew Bogut,2010,2019
1,101107,Marvin Williams,2010,2020
2,101108,Chris Paul,2010,2026


## 3A. Is the data tidy?

The observation should be one player in one regular season. We check for duplicate keys, and look at a player who was traded mid-season (Luka Dončić, Dallas → Lakers in February 2025) to see whether he has one row or one per team.

In [5]:
print('duplicate (season, season_type, player_id) rows:', stats_raw.duplicated(['season', 'season_type', 'player_id']).sum())
print('\nregular-season rows by number of teams played for:')
print(rs_raw['team_count'].value_counts().sort_index().to_string())

luka = players.loc[players['name'] == 'Luka Dončić', 'player_id'].iloc[0]
rs_raw[(rs_raw['player_id'] == luka) & (rs_raw['season'] == 2025)][['season', 'player_id', 'team_count', 'player_last_team_id', 'gp', 'min', 'pts']]

duplicate (season, season_type, player_id) rows: 0

regular-season rows by number of teams played for:
team_count
1    13259
2     1668
3      112
4        3
5        1


,season,player_id,team_count,player_last_team_id,gp,min,pts
19395,2025,1629029,2,1610612747,50,1768.55,1408


**What we found:** The table is tidy, with one row per player, season, and season type and no duplicate keys. **A traded player has one combined row for the season**: Dončić's 2024-25 row covers all 50 of his games with both teams, with `team_count` = 2 and his last team (the Lakers) recorded. So nothing is double counted, and per-game stats can be computed straight from the totals. Players who changed teams are in the `team_count` ≥ 2 rows.

Name and team abbreviation are in separate lookup tables (`players`, `team_seasons`), joined in section 4.

## 3B. Size of the data

In [6]:
print('rows, columns (both season types):', stats_raw.shape)
print('regular-season seasons available:', rs_raw['season'].min(), 'to', rs_raw['season'].max())
rs = rs_raw.query('@HISTORY_FIRST <= season <= @LAST_SEASON').copy()
rs.groupby('season').size().rename('players').to_frame().T

rows, columns (both season types): (21142, 18)
regular-season seasons available: 1997 to 2027


season,2013,2014,2015,2016,2017,2018,2019,2020,2021,2022,2023,2024,2025,2026
players,469,482,492,476,486,540,530,529,540,605,539,572,569,582


The source covers 1996-97 through the current 2026-27 season, which is still in progress and is dropped. We keep **2012-13 through 2025-26**: the 11 seasons in our window plus 3 earlier ones for the trends. About 470-600 players appear each regular season, and the number rises over time. That is more than 30 rosters of 15, because of two-way contracts (from 2017-18), 10-day contracts, and mid-season signings.

## 3C. Missing data

In [7]:
rs.isna().sum()

season                    0
season_type               0
player_id                 0
player_last_team_id     503
team_count                0
age                       0
gp                        0
min                       0
pts                       0
reb                       0
ast                       0
stl                       0
blk                       0
fga                       0
fta                       0
plus_minus                0
ts_pct                    0
draft_year             1604
dtype: int64

In [8]:
no_attempts = rs[(rs['fga'] + rs['fta']) == 0]
print('player-seasons with no shot attempts at all:', len(no_attempts))
print('their ts_pct values:', no_attempts['ts_pct'].value_counts(dropna=False).to_dict())
print('their games played:', no_attempts['gp'].describe()[['min', '50%', 'max']].to_dict())

player-seasons with no shot attempts at all: 35
their ts_pct values: {0.0: 35}
their games played: {'min': 1.0, '50%': 1.0, 'max': 4.0}


**What we found:**

- **`draft_year` is blank for undrafted players.** That is not missing data, since those players were never drafted. It only matters for the edge case in section 3E.
- **`player_last_team_id` is blank for a few rows** (about 30 player-seasons in 2015-16 to 2025-26; the blanks before 2013-14 are seasons we don't use). The source didn't record the player's last team. `last_team` is reference information, not a join key, so we leave it blank for those rows.
- **But `ts_pct` is 0 for players who never attempted a shot.** True shooting is points divided by attempts, so with zero attempts it is undefined, not 0%. These are 35 player-seasons, each with 1-4 games. Leaving them at 0 would look like the worst shooting in the league, so we set `ts_pct` to blank (`NaN`) for them in section 4.

## 3D. Outliers and suspicious entries

We check for impossible values (negative stats, more than 48 minutes per game, `ts_pct` that doesn't match its formula), look at the extremes, and compare a few rows with published stat lines.

In [9]:
rs['mpg'] = rs['min'] / rs['gp']
rs['ppg'] = rs['pts'] / rs['gp']
ts_calc = rs['pts'] / (2 * (rs['fga'] + 0.44 * rs['fta']))

print('rows with a negative stat:', (rs[['gp', 'min', 'pts', 'reb', 'ast', 'stl', 'blk', 'fga', 'fta']] < 0).any(axis=1).sum())
print('rows with gp = 0:', (rs['gp'] == 0).sum())
print('rows with more than 48 minutes per game:', (rs['mpg'] > 48).sum())
print('rows where ts_pct differs from its formula by more than 0.002:', ((ts_calc - rs['ts_pct']).abs() > 0.002).sum())
print('rows with more than 82 games:', (rs['gp'] > 82).sum())
print('rows with ts_pct above 1:', (rs['ts_pct'] > 1).sum(), '| their games played:', sorted(rs.loc[rs['ts_pct'] > 1, 'gp'].unique().tolist()))

rows with a negative stat: 0
rows with gp = 0: 0
rows with more than 48 minutes per game: 0
rows where ts_pct differs from its formula by more than 0.002: 0
rows with more than 82 games: 4
rows with ts_pct above 1: 8 | their games played: [1, 2, 3, 5]


In [10]:
over82 = rs[rs['gp'] > 82].merge(players[['player_id', 'name']], on='player_id')
over82[['season', 'name', 'gp', 'team_count']]

,season,name,gp,team_count
0,2014,Ramon Sessions,83,2
1,2015,Josh Smith,83,2
2,2023,Mikal Bridges,83,2
3,2024,Buddy Hield,84,2


In [11]:
top = (rs.merge(players[['player_id', 'name']], on='player_id')
         .sort_values('ppg', ascending=False).groupby('season').head(1)
         .sort_values('season')[['season', 'name', 'gp', 'ppg', 'mpg']])
top.round(1)

,season,name,gp,ppg,mpg
402,2013,Carmelo Anthony,67,28.7,37.0
543,2014,Kevin Durant,81,32.0,38.5
1041,2015,Russell Westbrook,67,28.1,34.4
1608,2016,Stephen Curry,79,30.1,34.2
2112,2017,Russell Westbrook,81,31.6,34.6
2696,2018,James Harden,72,30.4,35.4
3273,2019,James Harden,78,36.1,36.8
3844,2020,James Harden,68,34.3,36.5
4400,2021,Stephen Curry,63,32.0,34.2
5131,2022,Joel Embiid,68,30.6,33.8


In [12]:
known_lines = [            # (player, season, stat, published value)
    ('James Harden', 2019, 'ppg', 36.1),
    ('Russell Westbrook', 2017, 'ppg', 31.6),
    ('Stephen Curry', 2016, 'ppg', 30.1),
    ('Nikola Jokić', 2024, 'ppg', 26.4),
    ('Nikola Jokić', 2024, 'gp', 79),
]
named = rs.merge(players[['player_id', 'name']], on='player_id')
for name, season, stat, expected in known_lines:
    got = named.loc[(named['name'] == name) & (named['season'] == season), stat].iloc[0]
    print(f"{'OK   ' if abs(got - expected) <= 0.15 else 'CHECK'} {name} {season} {stat}: got {got:.2f}, published {expected}")

OK    James Harden 2019 ppg: got 36.13, published 36.1
OK    Russell Westbrook 2017 ppg: got 31.58, published 31.6
OK    Stephen Curry 2016 ppg: got 30.06, published 30.1
OK    Nikola Jokić 2024 ppg: got 26.39, published 26.4
OK    Nikola Jokić 2024 gp: got 79.00, published 79


**What we found:**

- **No impossible values.** No stat is negative, nobody has 0 games or more than 48 minutes per game, and `ts_pct` matches its formula on every row.
- **Four players have more than 82 games, and all of them changed teams** (`team_count` = 2; Mikal Bridges in 2022-23 and Buddy Hield in 2023-24 are in our window). A traded player can join a team that has played fewer games than his old one, so his total can go past 82. These are real.
- **`ts_pct` above 1 is real for tiny samples.** One made 3-pointer on one shot is a true shooting % of 1.5. All 8 such rows belong to players with 1-5 games. We keep these values; `rs_gp` shows how small the sample is.
- **The top scorer each season is the real scoring leader** (Curry, Westbrook, Harden, Embiid, Dončić, Gilgeous-Alexander, ...), and the published stat lines match. Stars are extreme but not errors, so no rows are removed.

## 3E. Years in the NBA: draft year vs first season

There are two ways to count experience: from the draft year (`season - draft_year`), or from the player's **first season in the stats table** (`season - first_season + 1`, where `first_season` is the earliest season he has a row, regular season or playoffs). They should agree for a player who joined the league the season after he was drafted.

*We don't use the `first_season` column of the `players` table: in the cached copy of that table it starts at 2009-10 for every player who debuted earlier (LeBron James and Chris Paul both show 2010), so it would undercount veterans.*

In [13]:
first_season = stats_raw.groupby('player_id')['season'].min().rename('first_season')
exp = (rs.query('@FIRST_SEASON <= season')
         .merge(first_season, on='player_id', how='left', validate='many_to_one')
         .merge(players[['player_id', 'name']], on='player_id', how='left', validate='many_to_one'))
exp['years_from_draft'] = exp['season'] - exp['draft_year']
exp['years_from_first_season'] = exp['season'] - exp['first_season'] + 1

print('rows with no first_season:', exp['first_season'].isna().sum())
print('rows where first_season is after the season itself:', (exp['first_season'] > exp['season']).sum())
diff = exp[exp['draft_year'].notna() & (exp['years_from_draft'] != exp['years_from_first_season'])]
print('drafted player-seasons where the two counts disagree:', len(diff), 'of', exp['draft_year'].notna().sum())
print('...draft count is higher in:', (diff['years_from_draft'] > diff['years_from_first_season']).sum())
(diff.query('season == 2026').sort_values('years_from_draft', ascending=False)
     [['name', 'draft_year', 'first_season', 'years_from_draft', 'years_from_first_season']].head(8))

rows with no first_season: 0
rows where first_season is after the season itself: 0
drafted player-seasons where the two counts disagree: 417 of 4605
...draft count is higher in: 417


,name,draft_year,first_season,years_from_draft,years_from_first_season
5923,Jonas Valanciunas,2011.0,2013,15.0,14
5958,Joel Embiid,2014.0,2017,12.0,10
5946,Thanasis Antetokounmpo,2014.0,2016,12.0,11
5961,Bogdan Bogdanović,2014.0,2018,12.0,9
5963,Nikola Jokić,2014.0,2016,12.0,11
5959,Dario Saric,2014.0,2017,12.0,10
5413,Guerschon Yabusele,2016.0,2018,10.0,9
5432,Isaiah Hartenstein,2017.0,2019,9.0,8


In [14]:
# The source starts in 1996-97 (end year 1997), so first_season can't be earlier than 1997.
early = exp[(exp['first_season'] == stats_raw['season'].min()) & (exp['draft_year'] + 1 < exp['first_season'])]
early[['season', 'name', 'draft_year', 'first_season', 'years_from_first_season']]

,season,name,draft_year,first_season,years_from_first_season
474,2016,Kevin Garnett,1995.0,1997,20


**What we found:**

- **Every player has a `first_season`, and none is after the season itself.**
- **When the two counts disagree (417 drafted player-seasons), the draft count is always the higher one.** Most are draft-and-stash players: Nikola Jokić, Bogdan Bogdanović, Dario Šarić, and Jonas Valančiūnas were drafted years before they came to the NBA, so counting from the draft year adds seasons they spent playing in Europe. A few are players who missed their first seasons injured (Joel Embiid was drafted in 2014 but first played in 2016-17).
- We use **`season - first_season + 1`**, so `years_in_nba` counts **seasons since the player's first NBA game**, and a rookie is in year 1. It also works for undrafted players, who have no draft year. For players like Embiid it counts from his debut rather than his signing, which is the closer match to on-court experience.
- **One edge case:** the source starts in 1996-97, so `first_season` can't be earlier than 1997. That cuts off the start of the career for a player who debuted before then and was still playing in 2015-16 (Kevin Garnett, who debuted in 1995-96). For those rows (Garnett's 2015-16 season in our window, plus a few history seasons used only for the trends) we use `draft_year + 1` as the first season instead, since these players joined the league right after the draft.

## 4. Clean the data and build the features

### 4A. Per-game stats and true shooting

Divide each season total by games played. `ts_pct` is set to blank for players with no shot attempts (section 3C).

In [15]:
for stat, total in [('ppg', 'pts'), ('rpg', 'reb'), ('apg', 'ast'), ('bpg', 'blk'), ('spg', 'stl'),
                    ('mpg', 'min'), ('plus_minus', 'plus_minus')]:
    rs[stat] = rs[total] / rs['gp']
rs['ts_pct'] = rs['ts_pct'].mask((rs['fga'] + rs['fta']) == 0)
print('ts_pct set to blank:', rs['ts_pct'].isna().sum())

ts_pct set to blank: 35


### 4B. Years in the NBA

In [16]:
rs['first_nba_season'] = rs['player_id'].map(first_season)
truncated = (rs['first_nba_season'] == stats_raw['season'].min()) & (rs['draft_year'] + 1 < rs['first_nba_season'])
rs.loc[truncated, 'first_nba_season'] = rs.loc[truncated, 'draft_year'] + 1
rs['years_in_nba'] = (rs['season'] - rs['first_nba_season'] + 1).astype(int)
print('rows using draft_year + 1:', truncated.sum())
rs.query('@FIRST_SEASON <= season')['years_in_nba'].describe()[['min', '50%', 'max']]

rows using draft_year + 1: 10


min     1.0
50%     4.0
max    23.0
Name: years_in_nba, dtype: float64

### 4C. 3-year averages

For each player and season we average his stats over the **three previous seasons** (season − 3 to season − 1), using the ones he played in. The current season is not included. `seasons_played_last_3` (0-3) records how many seasons went into the average. The averages are blank only when it is 0, which means rookies and players returning after three or more seasons away. The shared helper `prior_seasons_average()` in `nba_cleaning_utils.py` does the calculation.

These are regular-season averages only. A playoff average would need the player's team to make three postseasons in a row, so most rows would be blank. The Box +/- average is built in notebook 03 the same way.

In [17]:
trends = prior_seasons_average(rs, 'player_id', ['ppg', 'rpg', 'apg', 'mpg', 'gp'], n=3)
rs = rs.merge(trends, on=['player_id', 'season'], how='left', validate='one_to_one')

# Example: Klay Thompson missed 2019-20 and 2020-21 with injuries
klay = players.loc[players['name'] == 'Klay Thompson', 'player_id'].iloc[0]
rs[(rs['player_id'] == klay) & rs['season'].between(2017, 2024)][
    ['season', 'gp', 'ppg', 'seasons_played_last_3', 'avg_ppg_last_3', 'avg_gp_last_3']].round(1)

,season,gp,ppg,seasons_played_last_3,avg_ppg_last_3,avg_gp_last_3
2207,2017,78,22.3,3,20.7,79.3
2752,2018,73,20.0,3,22.0,78.3
3323,2019,78,21.5,3,21.5,77.0
5052,2022,32,20.4,1,21.5,78.0
5608,2023,69,21.9,1,20.4,32.0
6195,2024,77,17.9,2,21.1,50.5


In [18]:
window = rs.query('@FIRST_SEASON <= season')
window['seasons_played_last_3'].value_counts().sort_index().rename('player-seasons').to_frame().T

seasons_played_last_3,0,1,2,3
player-seasons,1146,918,797,3107


**What we found:** Klay Thompson shows how the window works. He had no rows for 2019-20 or 2020-21, so his 2021-22 average uses only 2018-19 (1 season: 21.5 PPG over 78 games), and his 2022-23 average uses only 2021-22 (20.4 PPG over 32 games). His 2023-24 average is back to 2 seasons. A player who missed seasons gets an average from the seasons he did play, and `seasons_played_last_3` records how many there were.

About half of player-seasons have all 3 prior seasons, and about 1 in 5 have none (mostly rookies). Those keep blank averages, which XGBoost handles natively.

### 4D. Keep the feature columns

We add the player's name and `name_key` (for name-based joins), and his end-of-season team as `last_team`. We drop the raw totals and the history seasons before 2015-16, and add the `rs_` prefix to the regular-season stats. `last_team` is for reference only. **Join on `nba_player_id`, not team**: a traded player has one row here but one salary row per team.

In [19]:
abbrev = team_seasons[['season', 'team_id', 'nba_abbrev']].rename(columns={'team_id': 'player_last_team_id', 'nba_abbrev': 'last_team'})
out = (rs.query('@FIRST_SEASON <= season')
         .merge(players[['player_id', 'name']], on='player_id', how='left', validate='many_to_one')
         .merge(abbrev, on=['season', 'player_last_team_id'], how='left', validate='many_to_one'))
out['name_key'] = out['name'].map(normalize_name)

rs_stats = ['gp', 'mpg', 'ppg', 'rpg', 'apg', 'bpg', 'spg', 'ts_pct', 'plus_minus']
rs_features = (out.rename(columns={'player_id': 'nba_player_id', 'name': 'player_name', **{c: f'rs_{c}' for c in rs_stats}})
                  [['season', 'nba_player_id', 'player_name', 'name_key', 'last_team', 'age', 'years_in_nba']
                   + [f'rs_{c}' for c in rs_stats]
                   + ['seasons_played_last_3', 'avg_ppg_last_3', 'avg_rpg_last_3', 'avg_apg_last_3',
                      'avg_mpg_last_3', 'avg_gp_last_3']]
                  .sort_values(['season', 'nba_player_id']).reset_index(drop=True))
rs_features.head()

,season,nba_player_id,player_name,name_key,last_team,age,years_in_nba,rs_gp,rs_mpg,rs_ppg,...,rs_bpg,rs_spg,rs_ts_pct,rs_plus_minus,seasons_played_last_3,avg_ppg_last_3,avg_rpg_last_3,avg_apg_last_3,avg_mpg_last_3,avg_gp_last_3
0,2016,101106,Andrew Bogut,andrew bogut,GSW,31.0,11,70,20.732286,5.357143,...,1.628571,0.471429,0.623,5.657143,3,6.484764,8.591884,2.150653,24.860806,55.333333
1,2016,101107,Marvin Williams,marvin williams,CHA,30.0,11,81,28.866235,11.703704,...,0.950617,0.716049,0.585,3.061728,3,7.926792,4.537355,1.172888,25.034233,72.333333
2,2016,101108,Chris Paul,chris paul,LAC,31.0,11,74,32.705923,19.540541,...,0.175676,2.054054,0.575,7.472973,3,18.376310,4.216935,10.199592,34.405272,71.333333
3,2016,101109,Raymond Felton,raymond felton,DAL,32.0,11,80,27.399938,9.537500,...,0.187500,0.850000,0.502,-0.837500,3,9.119207,2.260087,4.166589,24.916843,54.000000
4,2016,101111,Charlie Villanueva,charlie villanueva,DAL,31.0,11,62,10.726398,5.145161,...,0.241935,0.290323,0.473,-1.435484,3,5.888330,2.506960,0.459828,11.801084,51.000000


## 5. Validate the final table

In [20]:
f = rs_features
checks = {
    'one row per season + player': not f.duplicated(['season', 'nba_player_id']).any(),
    f'seasons are {FIRST_SEASON}-{LAST_SEASON}': f['season'].between(FIRST_SEASON, LAST_SEASON).all() and f['season'].nunique() == 11,
    'every row has a name, age, and years in the NBA': f[['player_name', 'name_key', 'age', 'years_in_nba']].notna().all().all(),
    'last_team is blank only where the source has no last team': f['last_team'].isna().sum() == out['player_last_team_id'].isna().sum(),
    'years in the NBA is at least 1': (f['years_in_nba'] >= 1).all(),
    'games played is at least 1': (f['rs_gp'] >= 1).all(),
    'minutes per game is 0-48': f['rs_mpg'].between(0, 48).all(),
    'ts_pct is blank only when there were no attempts': f['rs_ts_pct'].isna().sum() == ((out['fga'] + out['fta']) == 0).sum(),
    'averages are blank exactly when seasons_played_last_3 is 0': (f['avg_ppg_last_3'].isna() == (f['seasons_played_last_3'] == 0)).all(),
}
for name, ok in checks.items():
    print('PASS' if ok else 'FAIL', '-', name)
assert all(checks.values())

PASS - one row per season + player
PASS - seasons are 2016-2026
PASS - every row has a name, age, and years in the NBA
PASS - last_team is blank only where the source has no last team
PASS - years in the NBA is at least 1
PASS - games played is at least 1
PASS - minutes per game is 0-48
PASS - ts_pct is blank only when there were no attempts
PASS - averages are blank exactly when seasons_played_last_3 is 0


In [21]:
f.drop(columns=['season']).describe().T[['count', 'mean', 'min', '50%', 'max']].round(2)

,count,mean,min,50%,max
age,5968.0,26.33,19.00,26.00,43.00
years_in_nba,5968.0,5.36,1.00,4.00,23.00
rs_gp,5968.0,47.11,1.00,53.00,84.00
rs_mpg,5968.0,19.47,0.35,19.16,43.52
rs_ppg,5968.0,8.64,0.00,7.03,36.13
rs_rpg,5968.0,3.54,0.00,3.06,15.99
rs_apg,5968.0,1.95,0.00,1.35,11.74
rs_bpg,5968.0,0.39,0.00,0.28,3.83
rs_spg,5968.0,0.63,0.00,0.57,3.01
rs_ts_pct,5939.0,0.54,0.00,0.55,1.50


**What we found:** Every check passes. The only blanks are the expected ones: `rs_ts_pct` for players with no shot attempts, `last_team` where the source has no last team, and the 3-year averages for players with no regular-season row in the three previous seasons.

## 6. Save

In [22]:
out_path = PROCESSED / 'llimllib_player_regular_season_features.csv'
rs_features.to_csv(out_path, index=False)
print('saved', rs_features.shape, 'to', out_path)

saved (5968, 22) to ..\..\data-files\02-processed\llimllib_player_regular_season_features.csv


**Output:** `data-files/02-processed/llimllib_player_regular_season_features.csv`, **one row per player + season** (players with at least one regular-season game, 2015-16 to 2025-26). Keys: `season` (end year) + `nba_player_id`; also `name_key` for name-based joins. Notebook `999` combines it with the playoff stats (`02`) and Box +/- (`03`) into `player_stats_combined.csv`, which is the table to join onto the salary table.